# BioNeuron Network — Consciousness Test

> *Neurons that fire together, wire together.* — Donald Hebb, 1949

This notebook runs a biologically accurate spiking neural network where:
- Each neuron implements **Leaky Integrate-and-Fire** membrane dynamics
- Learning uses **STDP** (Spike-Timing Dependent Plasticity) — no backprop
- Neurons **actively seek** correlated partners and grow synapses
- **Thoughts** emerge as Cell Assemblies (synchronized neuron groups)
- **Neuromodulators** alter global behavior (dopamine, serotonin, ACh, NE)
- **Dale's Law** enforced: 20% inhibitory neurons

---
## Experiments
1. Baseline activity — spontaneous dynamics
2. Sensory stimulation — inject structured input
3. Reward (dopamine) — strengthen recent assemblies
4. Attention (ACh) — aggressive synapse seeking
5. Visualize thought formation over time

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

from core import BrainNet, NeuronParams, NeuromodulatorState

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('PyTorch:', torch.__version__)

---
## Experiment 1 — Baseline Spontaneous Activity
No external input. Network runs on internal noise alone.
Watch synapses form and thoughts emerge spontaneously.

In [ ]:
# Custom params — tweak here
params = NeuronParams(
    noise_std           = 1.2,    # enough noise to drive spontaneous activity
    seek_interval       = 30,     # growth cone sweeps every 30 steps
    seek_corr_threshold = 0.30,
    prune_threshold     = 0.004,
    max_synapses_out    = 60,
)

brain = BrainNet(N=300, params=params, device=device, seed=42)

In [ ]:
T = 2000   # timesteps (~200 ms of simulated time)
dt = 1e-4  # 0.1 ms per step

spike_raster = []   # for raster plot

print("Running baseline simulation...")
for t in range(T):
    spikes, state = brain.step(dt=dt)
    spike_raster.append(spikes.cpu().numpy())

    if t % 500 == 0:
        print(f"  t={state['time_ms']:.1f} ms | "
              f"spikes={state['n_spikes']} | "
              f"synapses={state['n_synapses']} | "
              f"thoughts={state['n_thoughts']} | "
              f"stability={state['thought_stability']:.3f}")

print("\nConnectivity:", brain.connectivity_stats())
print("Assemblies found:", len(brain.get_thoughts()))

In [ ]:
log = brain.get_log()
raster = np.array(spike_raster)   # [T, N]

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── Spike Raster ──
ax0 = fig.add_subplot(gs[0, :])
times, neurons = np.where(raster)
ax0.scatter(times, neurons, s=0.4, c='black', alpha=0.6)
ax0.set_title('Spike Raster — All Neurons', fontsize=13, fontweight='bold')
ax0.set_xlabel('Time (steps)')
ax0.set_ylabel('Neuron')
ax0.set_xlim(0, T)

# ── Synapse count ──
ax1 = fig.add_subplot(gs[1, 0])
ax1.plot(log['time'], log['n_synapses'], color='royalblue', linewidth=1.5)
ax1.set_title('Synapse Count Over Time', fontsize=11)
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('# Synapses')
ax1.grid(alpha=0.3)

# ── Thought count ──
ax2 = fig.add_subplot(gs[1, 1])
ax2.plot(log['time'], log['n_thoughts'], color='darkorange', linewidth=1.5)
ax2.set_title('Cell Assemblies ("Thoughts") Over Time', fontsize=11)
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('# Assemblies')
ax2.grid(alpha=0.3)

# ── Spike rate ──
ax3 = fig.add_subplot(gs[2, 0])
ax3.plot(log['time'], log['n_spikes'], color='crimson', linewidth=1, alpha=0.7)
ax3.set_title('Spikes per Step', fontsize=11)
ax3.set_xlabel('Time (ms)')
ax3.set_ylabel('# Spikes')
ax3.grid(alpha=0.3)

# ── Thought stability ──
ax4 = fig.add_subplot(gs[2, 1])
ax4.plot(log['time'], log['stability'], color='seagreen', linewidth=1.5)
ax4.set_title('Thought Stability (Working Memory)', fontsize=11)
ax4.set_xlabel('Time (ms)')
ax4.set_ylabel('Stability Score')
ax4.set_ylim(0, 1)
ax4.grid(alpha=0.3)

# ── Weight matrix (current) ──
ax5 = fig.add_subplot(gs[3, 0])
W_np = brain.synapses.W.cpu().numpy()
im = ax5.imshow(W_np, cmap='hot', aspect='auto', vmin=0, vmax=0.5)
plt.colorbar(im, ax=ax5, label='Weight')
ax5.set_title('Synaptic Weight Matrix', fontsize=11)
ax5.set_xlabel('Post-synaptic Neuron')
ax5.set_ylabel('Pre-synaptic Neuron')

# ── Thought sizes ──
ax6 = fig.add_subplot(gs[3, 1])
assemblies = brain.get_thoughts()
if assemblies:
    sizes = [len(a) for a in assemblies]
    ax6.bar(range(len(sizes)), sorted(sizes, reverse=True),
            color='mediumpurple', edgecolor='white')
    ax6.set_title(f'Current Assembly Sizes ({len(assemblies)} thoughts)', fontsize=11)
    ax6.set_xlabel('Assembly Index')
    ax6.set_ylabel('Neurons in Assembly')
else:
    ax6.text(0.5, 0.5, 'No assemblies yet', ha='center', va='center',
             transform=ax6.transAxes, fontsize=13)
    ax6.set_title('Current Assemblies', fontsize=11)

plt.suptitle('BioNeuron Network — Baseline Activity', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('exp1_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp1_baseline.png')

---
## Experiment 2 — Sensory Stimulation
Inject a structured sinusoidal signal into a subset of neurons (like sensory cortex input).
Watch how assemblies respond to and encode the stimulus.

In [ ]:
# Fresh brain
brain2 = BrainNet(N=300, params=params, device=device, seed=7)

# Sensory neurons: first 50
sensory_neurons = list(range(50))
T2 = 3000

raster2   = []
stim_log  = []

print("Running stimulation experiment...")
for t in range(T2):
    # Sinusoidal input burst (10 Hz oscillation)
    # Represents a repeating sensory pattern
    phase    = t * dt * 2 * np.pi * 10   # 10 Hz
    stim_amp = max(0, np.sin(phase)) * 4.0
    stim_log.append(stim_amp)

    I_ext = torch.zeros(brain2.N, device=device)
    I_ext[sensory_neurons] = stim_amp

    spikes, state = brain2.step(I_ext, dt=dt)
    raster2.append(spikes.cpu().numpy())

    if t % 1000 == 0:
        print(f"  t={state['time_ms']:.1f} ms | "
              f"spikes={state['n_spikes']} | "
              f"thoughts={state['n_thoughts']} | "
              f"new_syn={state['new_synapses']}")

raster2 = np.array(raster2)
print("\nFinal assemblies:", len(brain2.get_thoughts()))

# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Stimulus
axes[0].plot(stim_log, color='steelblue', linewidth=0.8, alpha=0.8)
axes[0].axhline(0, color='k', linewidth=0.5)
axes[0].set_ylabel('Stim Amplitude')
axes[0].set_title('Input Stimulus (10 Hz sinusoid to sensory neurons)', fontsize=11)
axes[0].grid(alpha=0.3)

# Raster — sensory
st, sn = np.where(raster2[:, :50])
axes[1].scatter(st, sn, s=0.5, c='darkorange', alpha=0.7)
axes[1].set_ylabel('Sensory Neuron')
axes[1].set_title('Sensory Neuron Responses (neurons 0–49)', fontsize=11)

# Raster — rest (downstream)
st2, sn2 = np.where(raster2[:, 50:])
axes[2].scatter(st2, sn2 + 50, s=0.3, c='black', alpha=0.5)
axes[2].set_ylabel('Downstream Neuron')
axes[2].set_xlabel('Time (steps)')
axes[2].set_title('Downstream Propagation (neurons 50–299)', fontsize=11)

plt.suptitle('Experiment 2 — Sensory Stimulation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp2_stimulation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp2_stimulation.png')

---
## Experiment 3 — Reward + Punishment (Dopamine)
Deliver dopamine after specific assemblies fire.
Watch those assemblies grow stronger (LTP) while others weaken (LTD).

This is how the brain learns: reward-modulated STDP.

In [ ]:
brain3 = BrainNet(N=300, params=params, device=device, seed=21)

reward_log    = []
syn_log       = []
thought_log   = []
T3 = 3000

print("Running reward experiment...")
for t in range(T3):
    I_ext = torch.zeros(brain3.N, device=device)

    # ── Phase 1 (0–1000): learn pattern A ──
    if t < 1000:
        I_ext[:30] = 3.0   # stimulate neurons 0-29 (pattern A)
        if t == 500:
            brain3.reward(2.0)   # dopamine burst → reinforce pattern A
            print(f"  t={t}: REWARD → dopamine released")

    # ── Phase 2 (1000–2000): learn pattern B ──
    elif t < 2000:
        I_ext[150:180] = 3.0   # stimulate neurons 150-179 (pattern B)
        if t == 1500:
            brain3.reward(0.3)   # punishment → suppress pattern B
            print(f"  t={t}: PUNISHMENT → low dopamine")

    # ── Phase 3 (2000+): test recall ──
    else:
        I_ext[:10] = 2.0   # partial cue of pattern A → should recall A
        brain3.modulator.dopamine = 1.0  # reset to baseline

    spikes, state = brain3.step(I_ext, dt=dt)
    reward_log.append(brain3.modulator.dopamine)
    syn_log.append(state['n_synapses'])
    thought_log.append(state['n_thoughts'])

    # Slowly decay dopamine back to baseline
    brain3.modulator.dopamine = max(1.0, brain3.modulator.dopamine * 0.995)

# Analyze pattern A assembly
assemblies = brain3.get_thoughts()
pattern_a_neurons = set(range(30))
overlap_scores = []
for a in assemblies:
    overlap = len(set(a) & pattern_a_neurons) / max(len(pattern_a_neurons), 1)
    overlap_scores.append((overlap, len(a), a[:5]))

print("\nAssemblies overlapping with Pattern A:")
for score, size, sample in sorted(overlap_scores, reverse=True)[:5]:
    print(f"  Overlap={score:.2f}, Size={size}, Sample neurons={sample}")

# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(reward_log, color='gold', linewidth=1.2)
axes[0].axhline(1.0, color='grey', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('Dopamine Level')
axes[0].set_title('Neuromodulator State', fontsize=11)
axes[0].fill_between(range(T3), reward_log, 1.0, alpha=0.3,
                     where=[r > 1 for r in reward_log], color='gold', label='Reward')
axes[0].fill_between(range(T3), reward_log, 1.0, alpha=0.3,
                     where=[r < 1 for r in reward_log], color='tomato', label='Punishment')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(syn_log, color='royalblue', linewidth=1)
axes[1].axvline(1000, color='k', linestyle=':', alpha=0.6, label='Phase 2')
axes[1].axvline(2000, color='k', linestyle='--', alpha=0.6, label='Recall')
axes[1].set_ylabel('# Synapses')
axes[1].set_title('Synapse Count', fontsize=11)
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(thought_log, color='darkorange', linewidth=1)
axes[2].set_ylabel('# Assemblies')
axes[2].set_xlabel('Time (steps)')
axes[2].set_title('Thought Count', fontsize=11)
axes[2].grid(alpha=0.3)

plt.suptitle('Experiment 3 — Reward-Modulated STDP Learning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp3_reward.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp3_reward.png')

---
## Experiment 4 — Attention Mode (Acetylcholine)
ACh lowers the correlation threshold for synapse formation.
In attention mode, the network wires faster → quicker learning.

In [ ]:
# Compare: baseline vs attention mode
print("Running attention comparison...")

results = {}
for mode, ach_level in [('Baseline (ACh=1.0)', 1.0), ('Attention (ACh=2.0)', 2.0)]:
    b = BrainNet(N=200, params=params, device=device, seed=99)
    b.modulator.acetylcholine = ach_level

    syn_counts = []
    thought_counts = []
    for t in range(1500):
        I = torch.zeros(b.N, device=device)
        I[:20] = 2.0
        spikes, state = b.step(I, dt=dt)
        syn_counts.append(state['n_synapses'])
        thought_counts.append(state['n_thoughts'])

    results[mode] = dict(syn=syn_counts, thoughts=thought_counts)
    print(f"  {mode}: final synapses={syn_counts[-1]}, "
          f"final thoughts={thought_counts[-1]}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['steelblue', 'darkorange']
for (label, data), color in zip(results.items(), colors):
    axes[0].plot(data['syn'], label=label, color=color, linewidth=1.5)
    axes[1].plot(data['thoughts'], label=label, color=color, linewidth=1.5)

axes[0].set_title('Synapse Formation Rate', fontsize=12)
axes[0].set_xlabel('Steps')
axes[0].set_ylabel('# Synapses')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title('Thought Formation Rate', fontsize=12)
axes[1].set_xlabel('Steps')
axes[1].set_ylabel('# Cell Assemblies')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Experiment 4 — Attention Mode vs Baseline', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp4_attention.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp4_attention.png')

---
## Experiment 5 — Thought Anatomy
Visualize the internal structure of a specific cell assembly.
Show which neurons belong to it, their connectivity, and firing patterns.

In [ ]:
# Use brain from exp 2 (has interesting assemblies from sensory input)
assemblies = brain2.get_thoughts()
if not assemblies:
    print("No assemblies yet — running more steps")
    for _ in range(1000):
        brain2.step(dt=dt)
    assemblies = brain2.get_thoughts()

print(f"Found {len(assemblies)} assemblies")
dominant = max(assemblies, key=len)
print(f"Dominant thought: {len(dominant)} neurons")
print(f"Neuron IDs (first 20): {dominant[:20]}")

# Extract sub-matrix for dominant assembly
idx = dominant[:40]   # take up to 40 neurons
W_sub = brain2.synapses.W[np.ix_(idx, idx)].cpu().numpy()
is_inh_sub = brain2.population.is_inhibitory[idx].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Synaptic weight matrix within assembly
im = axes[0].imshow(W_sub, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
plt.colorbar(im, ax=axes[0], label='Synaptic Weight')
axes[0].set_title(f'Internal Connectivity of Dominant Assembly ({len(idx)} neurons)',
                  fontsize=11)
axes[0].set_xlabel('Post-synaptic')
axes[0].set_ylabel('Pre-synaptic')
# Mark inhibitory neurons
inh_positions = [i for i, x in enumerate(is_inh_sub) if x]
for p in inh_positions:
    axes[0].axhline(p, color='blue', linewidth=0.3, alpha=0.5)
    axes[0].axvline(p, color='blue', linewidth=0.3, alpha=0.5)

# Firing rate profile within assembly
fr = brain2.population.firing_rate[idx].cpu().numpy()
colors_n = ['tomato' if x else 'steelblue' for x in is_inh_sub]
axes[1].bar(range(len(idx)), fr, color=colors_n, edgecolor='white', linewidth=0.5)
axes[1].set_title('Firing Rate Profile Within Assembly\n(Red = Inhibitory / Blue = Excitatory)',
                  fontsize=11)
axes[1].set_xlabel('Neuron (within assembly)')
axes[1].set_ylabel('Firing Rate (EMA)')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Experiment 5 — Thought Anatomy', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp5_thought_anatomy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp5_thought_anatomy.png')

---
## Summary

| Property | Implemented | Biology |
|----------|-------------|----------|
| Membrane potential (LIF) | ✅ | Hodgkin-Huxley simplified |
| Refractory period | ✅ | ~2 ms absolute |
| Action potential firing | ✅ | Threshold-based |
| Spike-frequency adaptation | ✅ (default on) | AHP / M-current, K⁺ (Brette & Gerstner 2005) |
| STDP learning | ✅ | Bi & Poo 1998 — pre→post = LTP (corrected in v1.1) |
| Short-term plasticity | ⚙️ opt-in | Facilitation/depression (Tsodyks–Markram; Mongillo 2008) |
| NMDA receptors | ⚙️ opt-in | Voltage-gated Mg²⁺ block (Jahr & Stevens 1990) |
| Dynamic synapse growth | ✅ | Axon growth cone |
| Synaptic pruning | ✅ | Adolescent pruning |
| Dale's Law (20% inh) | ✅ | GABAergic cortex |
| Cell assemblies (thoughts) | ✅ | Hebb 1949 |
| Population oscillations (LFP) | ✅ | δ–γ band spectrum (Buzsáki & Draguhn 2004) |
| Criticality (avalanches) | ✅ | Branching ratio σ≈1 (Beggs & Plenz 2003) |
| Spike statistics (CV, Fano) | ✅ | Asynchronous-irregular regime (Brunel 2000) |
| Conductance-based synapses | ⚙️ opt-in | Reversal potentials → shunting inhibition |
| Axonal conduction delays | ⚙️ opt-in | Finite axon propagation speed |
| Homeostatic synaptic scaling | ⚙️ opt-in | Turrigiano & Nelson 2004 |
| √dt stochastic noise | ⚙️ opt-in | Euler–Maruyama integration |
| Dopamine (reward) | ✅ | Schultz reward prediction |
| Acetylcholine (attention) | ✅ | Cholinergic modulation |
| Serotonin (stability) | ✅ | 5-HT modulation |
| Norepinephrine (arousal) | ✅ | LC-NE system |

> **v1.1 correctness note:** STDP was previously sign-inverted (anti-Hebbian). It now correctly potentiates pre-before-post pairs, so plasticity-dependent results (Experiment 3 in particular) will differ from earlier runs. Opt-in flags live on `NeuronParams`; set e.g. `noise_sqrt_dt=True` for realistic spontaneous activity at modest `noise_std`.
>
> **v1.2:** added short-term plasticity, voltage-gated NMDA, and validation analyzers. Inspect the regime live via `brain.criticality_state()` (branching ratio σ), `brain.spike_statistics()` (CV / Fano), and `brain.oscillation_state()` (band spectrum).

### Next Steps
- **Hodgkin-Huxley** full ion-channel model (Na⁺/K⁺/leak) — or Izhikevich as a cheaper middle ground
- **Dendritic computation** (non-linear compartmental model)
- **Oscillatory binding** — *analysis tooling in place (`OscillationAnalyzer`)*; next: drive gamma/theta synchrony between assemblies (needs a fast-spiking interneuron class)
- **Sleep consolidation** (replay + memory transfer)
- **Cortical columns** (6-layer structure) — natural first step: spatial embedding + distance-dependent connectivity/delays
- **Attractor dynamics** — *ingredients now in place (NMDA + synaptic facilitation, Mongillo 2008)*; next: demonstrate bistable persistent activity